# Hafta 2 — Gelişmiş RAG: İndeksleme ve Vektör Optimizasyonu

> [`src/rag_engine/`](../src/rag_engine/) **için kaynak not defteri.**

## Öğrenme hedefleri

1. Saf, tek parçalı RAG'in uzun teknik belgelerde neden başarısız olduğunu teşhis etmek.
2. Birim normalize gömmelerin geometrisini ve birim küre üzerinde kosinüs, nokta çarpımı ve Öklid uzaklığının eşdeğerliğini türetmek.
3. Ebeveyn-çocuk hiyerarşik parçalamayı ve cümle penceresi erişimini uygulamak; hangisinin ne zaman tercih edileceğini açıklamak.
4. `src/rag_engine/hnsw_native.py` içindeki sıfırdan uygulamayı kullanarak HNSW'yi seviye atama olasılığından açgözlü arama döngüsüne kadar adım adım incelemek.
5. Bir derlemi Chroma vektör deposuna almak ve erişimi ayrılmış sorgularla doğrulamak.

> **Not:** Örnek metinler ve sorgular, kullanılan gömme modeli (`bge-small-en-v1.5`) İngilizce olduğu için İngilizce bırakılmıştır.

## 1. Saf RAG neden başarısız olur

En basit RAG hattı:

```
belge → 500 token'lık parçalara böl → göm → sakla
sorgu → göm → kosinüse göre ilk k → birleştir → LLM
```

İki hata türü baskındır:

**(a) Uzun parçalarda gömme sadakati çöker.** Sentence-Transformer'lar (Reimers ve Gurevych, 2019) kabaca 50–200 token uzunluğundaki girdilerle eğitilmiştir. Parça uzunluğu arttıkça gömme, parçadaki belirli iddiaları artık ayırt edemeyen, ortalaması alınmış bir "konu" vektörüne dönüşür. Erişim kabalaşır.

**(b) Kısa parçalarda bağlam yetersiz kalır.** 100 token'lık bir parça sorguyla eşleşen iddianın tam kendisini içerebilir, ancak LLM'in doğru yanıt verebilmesi için çevredeki bağlama (tanımlar, öncüller, paragrafın geri kalanı) ihtiyacı vardır. Halüsinasyon fırlar.

Bu hedefler — **gömme sadakati** ve **bağlamsal yeterlilik** — zıt yönlere çeker. Hiyerarşik ve cümle penceresi erişimi bunları birbirinden *ayırır*.

## 2. Geometrik temel

Birim normlu $\mathbf{u}, \mathbf{v} \in \mathbb{R}^d$ vektörleri için:

$$
\cos(\mathbf{u}, \mathbf{v}) = \mathbf{u}^\top \mathbf{v},
\qquad
\|\mathbf{u} - \mathbf{v}\|_2^2 = 2 - 2\,\mathbf{u}^\top \mathbf{v}.
$$

Dolayısıyla birim küre üzerinde **kosinüs benzerliği, iç çarpım ve karesel Öklid uzaklığı aynı sıralamayı üretir**. `src/rag_engine/embeddings.py` dosyasının çıktıları normalize etmesinin nedeni budur: kosinüs için yapılandırılmış bir Chroma koleksiyonu ile L2 için yapılandırılmış bir Milvus koleksiyonunun sessizce farklı sonuç verdiği bütün bir hata sınıfını ortadan kaldırır.

In [ ]:
import numpy as np

def normalize(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

rng = np.random.default_rng(0)
u, v = normalize(rng.standard_normal((2, 8)).astype(np.float32))

cos_uv = float(u @ v)
l2_sq  = float(np.sum((u - v) ** 2))
print(f"cos(u,v) = {cos_uv:.4f}")
print(f"||u-v||² = {l2_sq:.4f}   ≟   2 - 2·cos = {2 - 2*cos_uv:.4f}")


## 3. Hiyerarşik (ebeveyn-çocuk) parçalama

**Fikir.** Her belgeyi büyük *ebeveynlere* (~1024 token, paragrafa hizalı), her ebeveyni de daha küçük *çocuklara* (~256 token, cümleye hizalı) bölün. Erişim kesinliği için çocukları indeksleyin; bağlam için LLM'e ebeveyn metnini döndürün.

Uygulama [`src/rag_engine/chunking.py`](../src/rag_engine/chunking.py) dosyasındadır.

In [ ]:
from src.rag_engine.chunking import HierarchicalChunker

doc = '''Retrieval-Augmented Generation (RAG) couples a parametric language model
with a non-parametric retriever. The retriever indexes a corpus of text passages
and, at query time, returns the top-k passages by similarity to the query. These
passages are concatenated into the prompt for the generator.

A common failure mode is *chunk geometry mismatch*. Embedding models like SBERT
were trained on short inputs. Their vector quality degrades as input length grows.
Long chunks therefore embed poorly and retrieval recall suffers.

At the same time, short chunks deprive the generator of the surrounding context
it needs to answer. A two-sentence excerpt may match the query exactly and yet
omit the antecedent the LLM needs to interpret it. This produces fluent but
incorrect answers — the classic hallucination signature.

Hierarchical chunking decouples these objectives. Each document is split into
large *parent* chunks for context and smaller *child* chunks for retrieval.
At inference time the child hit triggers parent expansion: the LLM sees the full
parent text, while the retriever ranks against the focused child embedding.'''

chunker = HierarchicalChunker(parent_chars=600, child_chars=200, child_overlap_chars=40)
parents, children = chunker.chunk(doc, doc_metadata={"doc_id": "rag-101"})

print(f"{len(parents)} ebeveyn, {len(children)} çocuk")
print("---")
print("İlk çocuk:", children[0].text[:160], "...")
print("Ait olduğu ebeveyn eşleşiyor mu:", parents[0].chunk_id == children[0].parent_id)


**Neden örtüşme.** Çocuklar bir sonraki parçaya küçük bir örtüşme taşır (`child_overlap_chars`). Bu, *sınır etkisini* hafifletir: yanıtı bir parça sınırının iki yanına yayılan bir sorgu aksi hâlde her iki tarafı da kaçırırdı. Örtüşme depolama maliyetini artırır ve daha fazla vektör gömülmesini gerektirir, ancak pratikte uzun teknik metinlerde ölçülebilir geri çağırma kazançları sağlar.

## 4. Cümle penceresi erişimi

**Fikir.** En yüksek gömme kesinliği için *tek tek cümleleri* indeksleyin. Erişim anında, isabeti LLM'e vermeden önce ±W komşu cümlelik bir pencereye genişletin.

Bu ikili bir stratejidir: indeksleme anında aşırı kesinlik, erişim anında bağlamsal genişletme. *Yanıtın* tek bir cümle, *bağlamın* ise bir paragraf olduğu soru-yanıt derlemlerinde son derece iyi çalışır.

In [ ]:
from src.rag_engine.chunking import SentenceWindowChunker

sw = SentenceWindowChunker(window=2)
sw_chunks = sw.chunk(doc, doc_metadata={"doc_id": "rag-101"})
print(f"{len(sw_chunks)} cümle parçası")
print("isabet:", sw_chunks[5].text)
print("---")
print("±2 genişletilmiş:", sw.expand_window(sw_chunks[5], {"rag-101": sw_chunks}))


## 5. Temel ilkelerden HNSW

Büyük derlemlerde kapsamlı kosinüs araması sorgu başına $O(N)$'dir — $N=10^3$ için sorun değil, $N=10^7$'de ölümcül. HNSW (Malkov ve Yashunin, 2018) beklenen $O(\log N)$ aramaya ulaşır.

### Seviye ataması

Yeni bir noktaya şu en üst katman atanır:

$$
\ell = \lfloor -\ln(u) \cdot m_L \rfloor, \quad u \sim U(0,1), \quad m_L = \frac{1}{\ln M}.
$$

Bu, katmanlar üzerinde geometrik bir dağılım üretir: kabaca $N/M^\ell$ nokta $\ell$ katmanına ulaşır. En üst katman seyrek bir küçük-dünya grafı gibi davranır; en alt katman yoğun bir kNN grafıdır.

In [ ]:
import math, random
random.seed(0)

M = 16
m_L = 1.0 / math.log(M)
levels = [int(math.floor(-math.log(random.random() + 1e-12) * m_L)) for _ in range(10_000)]

from collections import Counter
counts = Counter(levels)
for layer in sorted(counts):
    bar = "█" * int(counts[layer] * 60 / 10_000)
    print(f"katman {layer:2d}: {counts[layer]:6d}  {bar}")


Üstel azalma, açgözlü inişi logaritmik zamanlı yapan şeyin ta kendisidir: üst katmanlar az nokta içerir, bu yüzden arama yanıtın bulunduğu alt katmana doğru hızla daralır.

### Açgözlü arama

En üst katmandaki giriş noktasından başlayarak:

1. Geçerli düğümün tüm komşularına olan uzaklığı hesapla.
2. Herhangi bir komşu sorguya geçerli düğümden daha yakınsa ona atla.
3. Daha yakın komşu kalmayana kadar tekrarla.
4. Bir alt katmana in ve devam et.
5. Katman 0'da, nihai ilk-k için `ef` boyutlu bir öncelik kuyruğu aramasına genişlet.

In [ ]:
from src.rag_engine.hnsw_native import HNSWIndex

# Küçük bir indeks oluştur ve davranışını incele.
rng = np.random.default_rng(1)
N, d = 1000, 64
vectors = normalize(rng.standard_normal((N, d)).astype(np.float32))

index = HNSWIndex(dim=d, M=16, ef_construction=100, ef_search=50, seed=42)
for i, v in enumerate(vectors):
    index.add(v, payload={"id": i})

print(f"ulaşılan en üst katman: {index._max_layer}")
print(f"giriş noktası id: {index._entry_point}")

# Akıl sağlığı denetimi: bilinen bir noktayı ara. Aynı noktayı ≈ 0 uzaklıkta döndürmeli.
hit = index.search(vectors[42], k=3)
for dist, payload in hit:
    print(f"  id={payload['id']:4d}  dist={dist:.6f}")


In [ ]:
# Ayrılmış bir sorguda kaba kuvvetle karşılaştır.
query = normalize(rng.standard_normal(d).astype(np.float32))

# Kaba kuvvet gerçek değeri.
bf_dists = 1.0 - vectors @ query
bf_top = np.argsort(bf_dists)[:10]

# HNSW yaklaşık ilk 10.
hnsw_top = [p["id"] for _, p in index.search(query, k=10)]

print("recall@10:", len(set(bf_top.tolist()) & set(hnsw_top)) / 10)


Düzgün dağılımlı veride HNSW, `ef_search=50` ile tipik olarak $\geq 0.95$ recall@10 değerine ulaşır. `ef_search` ayarı baskın kaldıraçtır: daha yüksek değerler sorgu gecikmesini geri çağırma karşılığında takas eder.

## 6. Uçtan uca: Chroma'ya veri alma

Üretim yolu, aynı `VectorStore` arayüzünün arkasında Chroma'nın HNSW'sini (daha hızlı, C destekli bir uygulama) kullanır. Yukarıdaki sıfırdan indeks matematiği anlamak içindir; kodda şunu kullanırsınız:

In [ ]:
from src.rag_engine.embeddings import SentenceEncoder
from src.rag_engine.pipeline import HierarchicalRAG
from src.rag_engine.vector_stores import build_store
import tempfile

persist = tempfile.mkdtemp()
rag = HierarchicalRAG(
    chunker=HierarchicalChunker(parent_chars=600, child_chars=200, child_overlap_chars=40),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    child_store=build_store("chroma", collection="demo", persist_dir=persist),
    parent_lookup={},
)

# Üç kısa belge al.
docs = [
    ("hyde",     "HyDE generates a hypothetical answer and embeds it for retrieval. The synthetic document is closer to real answer documents than the query is."),
    ("rerank",   "Cross-encoders score query-document pairs jointly. They are slower than bi-encoders but much more accurate, making them ideal for second-stage re-ranking."),
    ("chunking", "Hierarchical chunking splits documents into parent and child chunks. Children are indexed for retrieval precision; parents are returned for context."),
]
for doc_id, text in docs:
    rag.ingest_document(text, metadata={"doc_id": doc_id, "title": doc_id})

# Sorgula.
for q in ["how does re-ranking work?", "what is HyDE?", "parent-child chunks"]:
    hits = rag.retrieve(q, k=2)
    print(f"\nS: {q}")
    for h in hits:
        print(f"  [{h.score:.3f}] {h.metadata.get('doc_id')}  {h.text[:80]}…")


## 7. Alıştırmalar

1. **Parça geometrisini ayarlayın.** Ayrılmış bir soru-yanıt veri kümesinde `(parent_chars, child_chars)` değerlerini bir ızgara üzerinde tarayın. recall@10 ile ortalama LLM yanıt doğruluğunu çizdirin. Pareto sınırı nerede?
2. **IVF-PQ uygulayın.** HNSW bir ANN ailesidir; Ters Dosya + Çarpım Nicemleme (Inverted File + Product Quantization) bir diğeridir. Oyuncak bir IVF-PQ indeksi uygulayın ve aynı veride bellek ayak izini HNSW ile karşılaştırın.
3. **Cümle penceresi ve hiyerarşik.** Stratejilerden birinin açıkça kazandığı bir derlem seçin. *Nedenini* açıklayın — yanıt, o derlemdeki tipik sorgu/yanıt uzunluğu oranına atıfta bulunmalıdır.

## 8. Çıkarımlar

- Parça geometrisi bir RAG sistemindeki en önemli tek hiperparametredir. Hiyerarşik ve cümle penceresi stratejileri gömme sadakati / bağlamsal yeterlilik ödünleşimini *ayırır*.
- **Tüm gömmeleri birim normalize edin.** Kosinüs, nokta çarpımı ve Öklid eşdeğer sıralamalara dönüşür ve arka uç uyumsuzluğu hatalarının bütün bir sınıfı ortadan kalkar.
- HNSW'nin beklenen $O(\log N)$ araması, basit bir seviye örnekleme formülüyle elde edilen geometrik bir katman doluluk dağılımına dayanır.
- `VectorStore` Protocol'ü, arka uç seçiminin mimari bir taahhüt değil, bir yapılandırma seçeneği olduğu anlamına gelir.

➡ Gelecek hafta *diğer* eksene saldırıyoruz: sorgu dönüşümü (HyDE) ve cross-encoder ile yeniden sıralama.